### Load dependencies

In [1]:
import spikeinterface.full as si
import numpy as np
import os
import NpxCodes.NpxUtils as NpxUtils

print(f"SpikeInterface version: {si.__version__}")

job_kwargs = dict(n_jobs=40, chunk_duration='1s', progress_bar=True)

SpikeInterface version: 0.103.0


In [2]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import seaborn as sns

font_path = r'../misc/Inter-4.1/extras/ttf/Inter-Light.ttf'
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = prop.get_name()
plt.rcParams['font.size'] = 11

custom_params = {"axes.spines.right": False, "axes.spines.top": False}
sns.set_theme(style="ticks", palette="colorblind", font_scale=1.2, rc=custom_params, font=prop.get_name())

### Load data

#### Single session

In [3]:
animal = 'Fa0156'
base_session_folder = rf"../data/{animal}/Day9_CntrlCntrlCntrl_Shank3BankA"

# Check if the directory exists before proceeding
if not os.path.isdir(base_session_folder):
    print(f"Error: The directory '{base_session_folder}' does not exist.")
    print("Please update the 'rec_folder' variable to your actual data path.")
    exit()

recording_folders = NpxUtils.get_rec_folders(base_session_folder)
print(f"Found {len(recording_folders)} recording folders.")
recording_folders.sort()
print(recording_folders)
recording = recording_folders[0]

Found 3 recording folders.
['../data/Fa0156/Day9_CntrlCntrlCntrl_Shank3BankA/1Cntrl/2024-08-20_23-53-22', '../data/Fa0156/Day9_CntrlCntrlCntrl_Shank3BankA/2Cntrl/2024-08-21_00-30-56', '../data/Fa0156/Day9_CntrlCntrlCntrl_Shank3BankA/3Cntrl/2024-08-21_01-11-31']


In [4]:
# --- 2. Load Raw Data with SpikeInterface ---
print(f"\nLoading data from: {recording}")
raw_rec = si.read_openephys(recording, stream_id='1')
print('Initial recording properties:\n', list(raw_rec.get_property_keys()))

sampling_frequency = raw_rec.get_sampling_frequency()
num_channels = raw_rec.get_num_channels()

print(f"Original sampling frequency: {sampling_frequency} Hz")
print(f"Original number of channels: {num_channels}")

raw_timestamps = raw_rec.get_times()
recording_duration = raw_timestamps[-1] - raw_timestamps[0]
first_timestamp = raw_timestamps[0]
last_timestamp = raw_timestamps[-1]
print(f"Recording duration: {recording_duration} s")
print(f"First timestamp: {first_timestamp} s")
print(f"Last timestamp: {last_timestamp} s")

#list available channels
channel_ids = raw_rec.get_channel_ids()
print(f"number of channels: {len(channel_ids)}")
#select five random channels
np.random.seed(0)
random_channel_ids = np.random.choice(channel_ids, size=10, replace=False)
print(f"Randomly selected channel IDs: {random_channel_ids}")


Loading data from: ../data/Fa0156/Day9_CntrlCntrlCntrl_Shank3BankA/1Cntrl/2024-08-20_23-53-22
Initial recording properties:
 ['gain_to_uV', 'offset_to_uV', 'physical_unit', 'gain_to_physical_unit', 'offset_to_physical_unit', 'channel_name', 'contact_vector', 'location', 'group', 'inter_sample_shift']
Original sampling frequency: 30000.0 Hz
Original number of channels: 384
Recording duration: 614.1847333333334 s
First timestamp: 5.7504 s
Last timestamp: 619.9351333333334 s
number of channels: 384
Randomly selected channel IDs: ['CH214' 'CH125' 'CH191' 'CH53' 'CH251' 'CH329' 'CH309' 'CH177' 'CH279'
 'CH295']


#### (Optional) For Concatenated data

In [ ]:
animal = 'Fa0156'
base_folder = rf'../data/{animal}/Day9_CntrlCntrlCntrl_Shank3BankA/NeuralData/'

recording_folders = NpxUtils.get_rec_folders(base_folder)
recording_folders

In [ ]:
recordings_list = []
for dir in recording_folders:
    recordings_list.append(si.read_openephys(dir,stream_id='1'))
recordings_list

In [ ]:
raw_rec = si.concatenate_recordings(recordings_list)
raw_rec

In [ ]:
# --- 2. Load Raw Data with SpikeInterface ---
print('Initial recording properties:\n', list(raw_rec.get_property_keys()))

sampling_frequency = raw_rec.get_sampling_frequency()
num_channels = raw_rec.get_num_channels()

print(f"Original sampling frequency: {sampling_frequency} Hz")
print(f"Original number of channels: {num_channels}")

raw_timestamps = raw_rec.get_times()
recording_duration = raw_timestamps[-1] - raw_timestamps[0]
first_timestamp = raw_timestamps[0]
last_timestamp = raw_timestamps[-1]
print(f"Recording duration: {recording_duration} s")
print(f"First timestamp: {first_timestamp} s")
print(f"Last timestamp: {last_timestamp} s")

#list available channels
channel_ids = raw_rec.get_channel_ids()
print(f"number of channels: {len(channel_ids)}")
#select five random channels
np.random.seed(0)
random_channel_ids = np.random.choice(channel_ids, size=10, replace=False)
print(f"Randomly selected channel IDs: {random_channel_ids}")

#### (Optional) (Troubleshoot) Check Probe Info

In [ ]:
probe = raw_rec.get_probe()
print(probe)

fig, ax = plt.subplots(figsize=(6, 8))
#si.plot_probe_map(raw_rec, ax=ax, with_channel_ids=False)
si.plot_probe_map(recording, ax=ax, with_channel_ids=False)
ax.set_ylim(-200, 5000)

#### (Optional) (Troubleshoot) Plot traces from random channels

In [ ]:
raw_traces = si.plot_traces(raw_rec, time_range=(10,50), channel_ids=random_channel_ids, return_in_uV=True, show_channel_ids=True)

In [ ]:
si.plot_traces(raw_rec, backend='matplotlib', time_range=(105,110), clim=(-50,50), order_channel_by_depth=True)

### Preprocessing

In [ ]:
# --- 1. Setup and Configuration ---
print(si.pp_names_to_functions.keys())

preprocessing_dict = {
    'phase_shift': {},
    'detect_and_remove_bad_channels': {},
    'highpass_filter' : {'freq_min': 500, 'dtype': 'float32'},
    'common_reference': {'operator': 'median', 'reference': 'global'},
}

preprocessing_pipeline = si.PreprocessingPipeline(preprocessing_dict)
preprocessing_pipeline

dict_keys(['filter', 'bandpass_filter', 'highpass_filter', 'notch_filter', 'gaussian_filter', 'normalize_by_quantile', 'scale', 'center', 'zscore', 'scale_to_physical_units', 'whiten', 'common_reference', 'phase_shift', 'detect_and_remove_bad_channels', 'detect_and_interpolate_bad_channels', 'rectify', 'clip', 'blank_saturation', 'silence_periods', 'remove_artifacts', 'zero_channel_pad', 'deepinterpolate', 'resample', 'decimate', 'highpass_spatial_filter', 'interpolate_bad_channels', 'depth_order', 'average_across_direction', 'directional_derivative', 'astype', 'unsigned_to_signed'])


PreprocessingPipeline: 	Raw Recording → phase_shift → detect_and_remove_bad_channels → highpass_filter → common_reference → Preprocessed Recording

In [7]:
print("\nStarting preprocessing...")
preproc_rec = si.apply_preprocessing_pipeline(raw_rec, preprocessing_dict)
print("Preprocessing complete.")


Starting preprocessing...
Preprocessing complete.


In [ ]:
print(si.get_motion_presets())
si.get_motion_parameters_preset('dredge')

In [ ]:
print("\nStarting DREDGE motion correction...")
dredged_rec = si.correct_motion(preproc_rec, preset='dredge', **job_kwargs)
print("DREDGE motion correction complete.")

In [ ]:
dredged_rec_save_folder = recording + '_01_preprocessed_AP'

print(f"\nSaving preprocessed AP data in binary format to: {dredged_rec_save_folder}")
dredged_rec = dredged_rec.save(folder=dredged_rec_save_folder, format='binary', overwrite=True, **job_kwargs)
print("Saved preprocessed AP data successfully.")

In [5]:
dredged_rec_save_folder = recording + '_01_preprocessed_AP'
dredged_rec = si.load(dredged_rec_save_folder)
dredged_rec

BinaryFolderRecording: 384 channels - 30.0kHz - 1 segments - 18,425,543 samples 
                       614.18s (10.24 minutes) - float32 dtype - 26.36 GiB

#### (Optional) (Troubleshoot) Plot atrace from random channels for sanity check

In [ ]:
si.plot_traces(dredged_rec, time_range=(10,15), channel_ids=random_channel_ids, return_in_uV=True, show_channel_ids=True)

In [ ]:
si.plot_traces(dredged_rec, backend='matplotlib', time_range=(1005,1010), clim=(-50,50), order_channel_by_depth=True)

### Spike sorting

In [ ]:
si.get_default_sorter_params('kilosort4')

In [8]:
print("\nStarting spike sorting with Kilosort4...")
ks4out_folder = recording + '_10_KS4out'
sorting = si.run_sorter('kilosort4', dredged_rec, folder=ks4out_folder, docker_image=False, verbose=True, **job_kwargs)
print("Kilosort4 spike sorting complete.")


Starting spike sorting with Kilosort4...


kilosort.run_kilosort:  
kilosort.run_kilosort: Computing preprocessing variables.
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: N samples: 18425543
kilosort.run_kilosort: N seconds: 614.1847666666666
kilosort.run_kilosort: N batches: 308
kilosort.run_kilosort: Preprocessing filters computed in 5.57s; total 5.57s
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage after preprocessing
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:     7.70 %
kilosort.run_kilosort: Mem used:     10.00 %     |      18.91 GB
kilosort.run_kilosort: Mem avail:    169.36 / 188.27 GB
kilosort.run_kilosort: ------------------------------------------------------
kilosort.run_kilosort: GPU usage:    `conda install pynvml` for GPU usage
kilosort.run_kilosort: GPU memory:   17.65 %     |      1.90   /    10.75 GB
kilosort.run_kilosort: Allocated:     0.08 %     |      0.01   /    10.75 GB
kilosort.

kilosort4 run time 745.65s
Kilosort4 spike sorting complete.


In [9]:
ks4out_folder = recording + '_10_KS4out'
sorting = si.load(ks4out_folder)
sorting

KiloSortSortingExtractor: 678 units - 1 segments - 30.0kHz

### Postprocessing of units

In [11]:
# --- 7. postprocessing and compute si metrics for units ---
sorting_analyzer = si.create_sorting_analyzer(sorting=sorting, recording=dredged_rec, sparse=True, format="memory", return_in_uV=True)

estimate_sparsity (no parallelization):   0%|          | 0/615 [00:00<?, ?it/s]

In [16]:
all_computable_extensions = sorting_analyzer.get_computable_extensions()
print(f"All computable extensions: {all_computable_extensions}")

All computable extensions: ['random_spikes', 'waveforms', 'templates', 'noise_levels', 'amplitude_scalings', 'correlograms', 'isi_histograms', 'principal_components', 'spike_amplitudes', 'spike_locations', 'template_metrics', 'template_similarity', 'unit_locations', 'quality_metrics']


In [20]:
compute_dict = {
    'random_spikes' : {'method': 'uniform', 'max_spikes_per_unit': 500},
    'waveforms' : {'ms_before': 1.0, 'ms_after': 2.0},
    'templates' : {'operators': ['average','median','std']},
    'noise_levels' : {},
    'correlograms' : {},
    'unit_locations': {},
    'spike_amplitudes' : {},
    'template_similarity' : {},
    'quality_metrics' : {},
    'template_metrics' : {},
}

In [21]:
sorting_analyzer.compute(compute_dict, **job_kwargs)

compute_waveforms (workers: 20 processes):   0%|          | 0/615 [00:00<?, ?it/s]

Compute : spike_amplitudes (workers: 20 processes):   0%|          | 0/615 [00:00<?, ?it/s]

/gpfs/soma_fs/home/radadia/miniforge3/envs/npx/lib/python3.10/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1066: UserWarning: Some units have too few spikes : amplitude_cutoff is set to NaN
  warnings.warn(f"Some units have too few spikes : amplitude_cutoff is set to NaN")
/gpfs/soma_fs/home/radadia/miniforge3/envs/npx/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/gpfs/soma_fs/home/radadia/miniforge3/envs/npx/lib/python3.10/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/gpfs/soma_fs/home/radadia/miniforge3/envs/npx/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/gpfs/soma_fs/home/radadia/miniforge3/envs/npx/lib/python3.10/site-packages/spikeinterface/qua

In [13]:
SNR = si.compute_snrs(sorting_analyzer=sorting_analyzer)

In [14]:
SNR

{0: 1.4355764364952932,
 1: 3.5501414961949957,
 2: 1.3334892346731264,
 3: 2.069889655926606,
 4: 2.1712611310982877,
 5: 3.4036510148529677,
 6: 2.0544845133500846,
 7: 2.5618905170733965,
 8: 1.4019517675597792,
 9: 2.4829695855296734,
 10: 0.8886663215029708,
 11: 1.723122179650171,
 12: 1.3912382344324978,
 13: 2.665831737731299,
 14: 1.625638835374493,
 15: 2.092542424257625,
 16: 5.274616592615364,
 17: 4.191970176846723,
 18: 1.4728913869927427,
 19: 7.530178853136848,
 20: 3.869173973042635,
 21: 1.6799288442689595,
 22: 3.0891493486440122,
 23: 2.7476570070016186,
 24: 2.879116257443674,
 25: 1.766777199541586,
 26: 2.978189282759422,
 27: 4.456309420108677,
 28: 3.5284972409553443,
 29: 2.4946925338609387,
 30: 8.83890084644042,
 31: 1.5551569796672415,
 32: 6.9893954715613775,
 33: 3.5265494191584663,
 34: 3.3395924664756027,
 35: 2.128448006349652,
 36: 1.6723655493419434,
 37: 4.439239591161168,
 38: 3.8054765916968867,
 39: 4.8698370444194925,
 40: 13.848548637234659,
 4

### Curation using spikeinterface

In [22]:
available_extension_names = sorting_analyzer.get_loaded_extension_names()
print(available_extension_names)

['random_spikes', 'noise_levels', 'correlograms', 'waveforms', 'templates', 'unit_locations', 'template_similarity', 'template_metrics', 'spike_amplitudes', 'quality_metrics']


In [23]:
print(si.get_quality_metric_list())
print(si.get_quality_pca_metric_list())

['num_spikes', 'firing_rate', 'presence_ratio', 'snr', 'isi_violation', 'rp_violation', 'sliding_rp_violation', 'amplitude_cutoff', 'amplitude_median', 'amplitude_cv', 'synchrony', 'firing_range', 'drift', 'sd_ratio', 'noise_cutoff']
['isolation_distance', 'l_ratio', 'd_prime', 'nearest_neighbor', 'nn_isolation', 'nn_noise_overlap', 'silhouette']


In [24]:
qm_ext = sorting_analyzer.get_extension("quality_metrics")
metrics = qm_ext.get_data()

metrics

,num_spikes,firing_rate,presence_ratio,snr,isi_violations_ratio,isi_violations_count,rp_contamination,rp_violations,sliding_rp_violation,amplitude_cutoff,amplitude_median,amplitude_cv_median,amplitude_cv_range,sync_spike_2,sync_spike_4,sync_spike_8,firing_range,sd_ratio,noise_cutoff,noise_ratio
0,2216,3.608035,1.0,1.480488,0.416906,10.0,0.000000,0.0,0.145,0.001655,-5.351254,0.768948,0.263183,0.110560,0.000451,0.000000,3.40,1.013160,-0.650147,0.038302
1,10367,16.879285,1.0,3.555613,0.236207,124.0,0.064972,22.0,0.050,0.000112,-13.501637,0.376336,0.122540,0.069548,0.000193,0.000096,10.58,1.298492,-0.208792,0.115415
2,2644,4.304893,1.0,1.271504,0.380713,13.0,0.044938,1.0,0.105,0.000143,-5.660307,0.917909,0.362377,0.106278,0.000378,0.000000,3.60,1.080270,-0.140325,0.083275
3,139,0.226316,1.0,2.069890,0.000000,0.0,0.000000,0.0,NaN,NaN,-9.438003,NaN,NaN,0.107914,0.000000,0.000000,0.60,1.116095,-0.249714,0.087500
4,520,0.846651,1.0,2.016531,0.757131,1.0,0.000000,0.0,NaN,0.007988,-3.622253,1.696862,0.339183,0.101923,0.000000,0.000000,1.40,2.690135,-0.297393,0.068966
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
673,196,0.319122,1.0,3.020501,0.000000,0.0,0.000000,0.0,NaN,NaN,-9.799177,NaN,NaN,0.107143,0.000000,0.000000,1.00,1.905754,-0.082679,0.051282
674,745,1.212990,1.0,4.225186,1.106589,3.0,1.000000,2.0,NaN,0.002797,-23.755840,0.716883,0.396125,0.073826,0.000000,0.000000,5.59,2.323837,-0.015818,0.187500
675,2523,4.107884,1.0,4.368245,0.321620,10.0,0.280577,5.0,0.340,0.000282,-14.472726,0.412385,0.886744,0.085216,0.000396,0.000000,10.79,2.330897,0.253924,0.063791
676,1239,2.017308,1.0,13.269403,0.133363,1.0,0.225461,1.0,NaN,0.004375,-32.493530,0.276140,0.238872,0.071832,0.000000,0.000000,6.99,2.723043,-0.604690,0.082222


In [ ]:
sparsity = si.compute_sparsity(sorting_analyzer, method = "radius", radius_um = 30)

In [ ]:
%matplotlib widget
si.plot_unit_templates(sorting_analyzer,templates_percentile_shading=None,backend = "ipywidgets")

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


AppLayout(children=(HBox(children=(Checkbox(value=False, description='same axis'), Checkbox(value=True, descri…

Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


In [28]:
si.plot_quality_metrics(sorting_analyzer, include_metrics=["amplitude_cutoff", "isi_violations_ratio", "snr"],  backend="ipywidgets")

AppLayout(children=(UnitSelector(children=(Label(value='Units', layout=Layout(justify_content='center')), Sele…

In [29]:
isi_viol_thresh = 0.5
amp_cutoff_thresh = 0.2
our_query = f"amplitude_cutoff < {amp_cutoff_thresh} & isi_violations_ratio < {isi_viol_thresh}"
print(our_query)

amplitude_cutoff < 0.2 & isi_violations_ratio < 0.5


In [32]:
keep_units = metrics.query(our_query)
keep_unit_ids = keep_units.index.values
print(f" {len(keep_unit_ids)} units will pass quality filtering.")

 323 units will pass quality filtering.


In [34]:
sorting_auto = sorting.select_units(keep_unit_ids)
print(f"Number of units before curation: {len(sorting.get_unit_ids())}")
print(f"Number of units after curation: {len(sorting_auto.get_unit_ids())}")

auto_curated = sorting_analyzer.select_units(keep_unit_ids)

Number of units before curation: 678
Number of units after curation: 323


In [35]:
si.plot_quality_metrics(auto_curated, include_metrics=["amplitude_cutoff", "isi_violations_ratio", "snr"], backend="ipywidgets")

AppLayout(children=(UnitSelector(children=(Label(value='Units', layout=Layout(justify_content='center')), Sele…

In [ ]:
# create a folder to save the auto-curated phy results
curated_for_phy_folder = recording + '_11_KS4_curated_for_phy'
# export the auto-curated results to phy format
si.export_to_phy(auto_curated, output_folder=curated_for_phy_folder, compute_amplitudes=True, compute_pc_features=True, copy_binary=False,**job_kwargs)

Fitting PCA:   0%|          | 0/323 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/323 [00:00<?, ?it/s]

extract PCs (workers: 20 processes):   0%|          | 0/615 [00:00<?, ?it/s]

Run:
phy template-gui  /gpfs/soma_fs/nmr/NMR_group_data/Hemanya/data/Fa0156/Day9_CntrlCntrlCntrl_Shank3BankA/1Cntrl/2024-08-20_23-53-22_20_KS4_curated_for_phy/params.py
